In [13]:
from httpx import request
import torch
from sentence_transformers import SentenceTransformer,CrossEncoder
from transformers import AutoTokenizer,AutoModelForCausalLM,PreTrainedModel,PreTrainedTokenizer
import re
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
import pymupdf
import faiss
import numpy as np
import os 
import requests

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/himanshuvyas/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [3]:
#--------Encoder Model-------!
encoder_Tokenizer:PreTrainedTokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5")
encoder_Model:PreTrainedModel = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5',trust_remote_code=True).to(device=device)

#------Decoder LLm------!
decoder_Tokenizer:PreTrainedTokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
decoder_LLM:PreTrainedModel = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct',dtype=torch.float16).to(device=device)

#-------Reranking Model----!
reranking_model = CrossEncoder('BAAI/bge-reranker-base',device=device)

<All keys matched successfully>
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7395.67it/s]


In [4]:
def extract_pdf(path):
    pdf = pymupdf.open(path)
    full_text = ""
    for page in pdf:
        full_text += page.get_text() + '\n'

    full_text = re.sub(r'-\n', '', full_text) #Removes broken hyphenated words across lines. (transf-ormer = transformer)
    full_text = re.sub(r'(?<!\n)\n(?!\n)', ' ', full_text) #This fixes broken sentences. next char or previous char shouldn't be in next line
    full_text = re.sub(r' {2,}', ' ', full_text) #Removes multiple spaces.

    return full_text.strip()

In [5]:
def parent_child_chunk(text,parent_size,child_size,parent_overlap=20,child_overlap=5):
    sentence = sent_tokenize(text=text)
    chunk_token_len = [len(decoder_Tokenizer.encode(sent,add_special_tokens=False)) for sent in sentence]
    
    #------Parent chunk process-------!
    parent_chunks=[]
    parent_current_chunk=[]
    parent_chunk_len=0
    for sent,token_len in zip(sentence,chunk_token_len):
        if parent_chunk_len + token_len <= parent_size:
            parent_current_chunk.append(sent)
            parent_chunk_len += token_len
        else:
            if parent_current_chunk:
                parent_chunks.append(" ".join(parent_current_chunk))
            parent_current_chunk = parent_current_chunk[-parent_overlap:] + [sent]
            parent_chunk_len = sum(chunk_token_len[-parent_overlap:]) + token_len
    if parent_current_chunk:
       parent_chunks.append(" ".join(parent_current_chunk)) 
    
    #------Child chunk process-------!
    for parent_idx,parent in enumerate(parent_chunks):
        parent_sent = sent_tokenize(parent)
        parent_chunk_token_len = [len(encoder_Tokenizer.encode(sent,add_special_tokens=False)) for sent in parent_sent]

        child_chunks=[]
        child_current_chunk=[]
        child_chunk_len=0
        for sent,token_len in zip(parent_sent,parent_chunk_token_len):
            if child_chunk_len + token_len <= child_size:
                child_current_chunk.append(sent)
                child_chunk_len += token_len
            else:
                if child_current_chunk:
                    child_chunks.append(" ".join(child_current_chunk))
                child_current_chunk = child_current_chunk[-child_overlap:] + [sent]
                child_chunk_len = sum(parent_chunk_token_len[-child_overlap:]) + token_len
        if child_current_chunk:
            child_chunks.append(" ".join(child_current_chunk))

        #-----Result----!
        results = []
        for child_idx,child in enumerate(child_chunks):
            results.append({
                'parent_text':parent,
                'child_text':child,
                'parent_idx':parent_idx,
                'child_id':f'{parent_idx}_{child_idx}'
            })
            
    return results

Encoder:

In [ ]:
#-----Setup faiss and vector structure------!
def setupVectorDB(results):
    child_text= [c['child_text'] for c in results]

    prefix = ['search_document: ' + t for t in child_text]
    embeddings = encoder_Model.encode(inputs=prefix,batch_size=32,normalize_embeddings=True,show_progress_bar=True,device=device)

    #-------Build faiss index-----!
    dim = embeddings.shape[1] # Here the shpae has batch and featrues, and we need features to create vector structure
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype(np.float32))
    print(f'Number of total vectors: {index.ntotal}')
    return index,results

In [7]:
def Retrieve_Answer(index:faiss.IndexFlatIP,question,top_k,results):
    query = ['search_query: '+question]
    query_vector = encoder_Model.encode(query,normalize_embeddings=True).astype(np.float32)

    scores,indices = index.search(query_vector,top_k)
    
    answers = []
    for score,idx in zip(scores[0],indices[0]):
        answers.append({
            'score':score,
            'child_text':results[idx]['child_text'],
            'parent_text':results[idx]['parent_text']
        })
    
    return answers

Reranking:

In [8]:
def Answer_Reranking(top_k,results,question):
    pairs = [(question,answer['child_text']) for answer in results]
    score = reranking_model.predict(inputs=pairs)

    for i,result in enumerate(results):
        result['rerank_score'] = score[i]

    reranked = sorted(results,lambda x:x['rerank_score'],reverse=True)
    return reranked[:top_k]

Decoder LLM Model:

In [ ]:
def final_Answer(text):
    messages = [
        {
            "role":"system","content":f"""You are an Question-Answer expert, Who answer in formal and professional language from the provided context."""
        },
        {
            "role":"user","content":f"""You are an Question-Answer expert, Who answer in formal and professional language from the provided context."""
        }
    ]

    prompt = decoder_Tokenizer.apply_chat_template(conversation=messages,add_generation_prompt=True,tokenize=False)
    inputs = decoder_Tokenizer(text=prompt,return_tensors='pt').to(device=device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        output = decoder_LLM.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.1,
            top_p=0.8,
            pad_token_id=decoder_Tokenizer.eos_token_id
        )
    new_token = output[0][input_len:]
    decoded = decoder_Tokenizer.decode(new_token,skip_special_tokens=True)
    return decoded.strip()

LLM Api:

In [14]:
def Get_AnswerLLM(question,context,API_Key,url):
    header= {
        'Authorization':f"Bearer {API_Key}",
        'Content-Type':'Application/json'
    }

    payload = {
        'model':'meta/llama-3.2-1b-instruct',
        'messages':[
            {
                'role':'user',
                'content':f""" You are an Question-Answer expert,  Who answer in formal and professional language from the provided context.
                Answer the question from the provided context only, Don't use outside knowledge or references.
                If answer is not in the provided context, Say: "I can't find the answer, Please be more specific".

                Context:
                {context}
                """
            }
        ],
        'temperature':0.1,
        'max_tokens':100,
        'top_p':0.8,
        'stream':False,
        'frequency_penalty':0, # Penalty for the words that keep repeating like: Python is great, python uses less memory. Python requires structure. (1.0 makes it difficult to use same words again & again) 
        'presence_penalty':0 #it checked that if alteast word present once
    }

    #-------Get Response------!
    request = requests.post(url=url,headers=header,json=payload)
    response = request.json()

    if request.status_code != 200:
        return f"Request Failed ({request.status_code}): {response}"

    if "error" in response:
        return f"API Error: {response['error'].get('message')}"

    return (
    response
    .get("choices", [{}])[0]
    .get("message", {})
    .get("content", "No answer returned.")
  )

